# 02 — Tableau Dashboard Preparation

This notebook creates clean, analysis-ready CSV tables for a Tableau dashboard. Run `01_sql_pipeline_analysis.ipynb` first or use the same source file at `data/sales_pipeline.csv`.

It does not fabricate dashboard values; all tables are calculated from the provided dataset.

In [ ]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("data/sales_pipeline.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError("Add your source dataset at data/sales_pipeline.csv before running this notebook.")

df = pd.read_csv(DATA_PATH)
# Rename your source columns here if needed:
# df = df.rename(columns={"Deal Stage": "stage", "Close Value": "revenue"})

required = {"opportunity_id", "sales_agent", "product", "stage", "revenue"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

df["stage"] = df["stage"].astype("string").str.strip()
df["stage_norm"] = df["stage"].str.lower()
df["revenue"] = pd.to_numeric(df["revenue"], errors="coerce").fillna(0)
df["is_won"] = df["stage_norm"].eq("won")
df["is_lost"] = df["stage_norm"].eq("lost")
df["is_closed"] = df["is_won"] | df["is_lost"]
df["is_active"] = ~df["is_closed"]

out_dir = Path("data/tableau_exports")
out_dir.mkdir(parents=True, exist_ok=True)
print(f"Loaded {len(df):,} opportunities.")

## 1. Executive KPI cards

In [ ]:
closed_count = int(df["is_closed"].sum())
won_count = int(df["is_won"].sum())
kpis = pd.DataFrame([{
    "total_opportunities": int(len(df)),
    "active_deals": int(df["is_active"].sum()),
    "closed_opportunities": closed_count,
    "won_opportunities": won_count,
    "closed_win_rate_pct": round(100 * won_count / closed_count, 2) if closed_count else None,
    "won_revenue": float(df.loc[df["is_won"], "revenue"].sum())
}])
kpis.to_csv(out_dir / "overall_kpis.csv", index=False)
display(kpis)

## 2. Agent performance

In [ ]:
agent = (
    df.groupby("sales_agent", dropna=False)
      .agg(
          total_opportunities=("opportunity_id", "size"),
          won_opportunities=("is_won", "sum"),
          lost_opportunities=("is_lost", "sum"),
          active_deals=("is_active", "sum"),
          won_revenue=("revenue", lambda s: float(s[df.loc[s.index, "is_won"]].sum()))
      )
      .reset_index()
)
agent["closed_opportunities"] = agent["won_opportunities"] + agent["lost_opportunities"]
agent["win_rate_pct"] = (
    100 * agent["won_opportunities"] / agent["closed_opportunities"].where(agent["closed_opportunities"] != 0)
).round(2)
agent = agent.sort_values(["win_rate_pct", "closed_opportunities"], ascending=[False, False])
agent["performance_rank"] = range(1, len(agent) + 1)
agent.to_csv(out_dir / "agent_performance.csv", index=False)
display(agent.head(30))

## 3. Product performance

In [ ]:
product = (
    df.groupby("product", dropna=False)
      .agg(
          total_opportunities=("opportunity_id", "size"),
          active_deals=("is_active", "sum"),
          won_opportunities=("is_won", "sum"),
          lost_opportunities=("is_lost", "sum"),
          won_revenue=("revenue", lambda s: float(s[df.loc[s.index, "is_won"]].sum()))
      )
      .reset_index()
)
product["closed_opportunities"] = product["won_opportunities"] + product["lost_opportunities"]
product["win_rate_pct"] = (
    100 * product["won_opportunities"] / product["closed_opportunities"].where(product["closed_opportunities"] != 0)
).round(2)
product = product.sort_values("won_revenue", ascending=False)
product.to_csv(out_dir / "product_performance.csv", index=False)
display(product)

## 4. Monthly won-revenue trend (if `close_date` exists)

In [ ]:
if "close_date" in df.columns:
    dates = pd.to_datetime(df["close_date"], errors="coerce")
    trend_source = df.loc[df["is_won"] & dates.notna()].copy()
    trend_source["month"] = dates.loc[trend_source.index].dt.to_period("M").astype(str)
    monthly = (
        trend_source.groupby("month", as_index=False)
        .agg(won_opportunities=("opportunity_id", "size"), won_revenue=("revenue", "sum"))
        .sort_values("month")
    )
    monthly.to_csv(out_dir / "monthly_revenue.csv", index=False)
    display(monthly)
else:
    print("No close_date column found; skipping monthly trend export.")

## Tableau dashboard layout

Suggested sheets:
- **Executive overview:** total opportunities, active deals, closed win rate, won revenue.
- **Agent performance:** ranked horizontal bars for win rate, with closed opportunity count in tooltips.
- **Product performance:** won revenue bars and win-rate comparison.
- **Monthly trend:** won revenue over time, if `close_date` is available.

Use filters for product, sales agent, and stage. Keep the win-rate denominator limited to `Won + Lost` opportunities.

In [ ]:
print("Exported dashboard tables:")
for path in sorted(out_dir.glob("*.csv")):
    print(f"- {path.name}")